# SkyGuard AI - Unsupervised ML Training
This notebook trains unsupervised models (Isolation Forest and LSTM Autoencoder) on clean historical data to learn normal weather patterns. It uses a strict chronological split to prevent temporal leakage.

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import sys

# Kaggle Path Handling
if os.path.exists('/kaggle/input'):
    dataset_dirs = glob.glob('/kaggle/input/*')
    if dataset_dirs:
        BASE_DIR = dataset_dirs[0] # Usually /kaggle/input/skyguard-repo
        sys.path.append(BASE_DIR)
        print(f"Running in Kaggle. BASE_DIR: {BASE_DIR}")
    else:
        raise ValueError("Please attach the repository as a Kaggle Dataset!")
else:
    BASE_DIR = '.'
    sys.path.append(os.path.abspath(BASE_DIR))
    print("Running locally.")

from src.models.iforest_model import MultivariateIForest
from src.models.lstm_ae_model import TemporalLSTMAE
from src.qc.physics_qc import PhysicsEngine

print("Libraries loaded.")

## 1. Load Data & Chronological Split

In [ ]:
# Load clean historical datasets
csv_files = glob.glob(os.path.join(BASE_DIR, 'data/raw/open_meteo/*2021_2024_hourly.csv'))
print(f"Found {len(csv_files)} clean station files.")

dfs = [pd.read_csv(f) for f in csv_files]
df = pd.concat(dfs, ignore_index=True)
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Add Physics features (needed for IForest)
phys = PhysicsEngine()
df = phys.add_derived_features(df)

# Chronological Split (Train: 2021-2023, Val: 2024)
train_df = df[df['timestamp'] < '2024-01-01'].copy()
val_df = df[df['timestamp'] >= '2024-01-01'].copy()

print(f"Total records: {len(df)}")
print(f"Training records: {len(train_df)}")
print(f"Validation records: {len(val_df)}")

## 2. Train Isolation Forest (Unsupervised)

In [ ]:
print("Training Multivariate Isolation Forest...")
iforest = MultivariateIForest(contamination=0.01)
iforest.train(train_df)
print("IForest training complete.")

## 3. Train LSTM Autoencoder (Unsupervised)

In [ ]:
print("Training Temporal LSTM Autoencoder...")
lstm_ae = TemporalLSTMAE(sequence_length=12, latent_dim=16)
history = lstm_ae.train(train_df, epochs=20, batch_size=256)
print("LSTM training complete.")

## 4. Export Artifacts

In [ ]:
out_dir = '/kaggle/working/models' if os.path.exists('/kaggle/input') else 'models'
os.makedirs(out_dir, exist_ok=True)

# Export exactly matching the application's expectation
iforest.save(os.path.join(out_dir, 'iforest_v2.pkl'))
lstm_ae.save(os.path.join(out_dir, 'lstm_ae_v2'))

print(f"Models exported successfully to {out_dir}")